# Αναζήτηση: Σημασιολογική Ανάκτηση

<img src="https://drive.google.com/uc?export=view&id=1M6xjXXC06apFzG42ga1Fom2A_C1SfisX" width="750">

Στο προηγούμενο notebook μελετήσαμε τις κλασικές μεθόδους ανάκτησης που λέγονται επίσης και "sparse retrieval" (BoW/TF‑IDF/BM25). Στην ουσία, όλες οι μέθοδοι που έχουμε εξετάσει βασίζονται σε λέξεις-κλειδιά, καθώς και σε αποδοτικούς τρόπους υλοποίησής τους (όπως το inverted index).

Τώρα περνάμε σε ένα διαφορετικό πρότυπο αναζήτησης, βασισμένο σε μεγάλα γλωσσικά μοντέλα και σημασιολογικές ενσωματώσεις (semantic embeddings): τη σημασιολογική αναζήτηση.

**Σημασιολογική Αναζήτηση**: μια εναλλακτική προσέγγιση που:
- Χρησιμοποιεί νευρωνικά δίκτυα για να μάθει σημασιολογικές αναπαραστάσεις κειμένου
- Μπορεί να ταιριάξει έγγραφα με βάση το νόημα (όχι μόνο κοινές λέξεις)
- Διαχειρίζεται φυσικά συνώνυμα, παραφράσεις και συναφείς έννοιες
- Απαιτεί συνήθως περισσότερους υπολογιστικούς πόρους (υπολογισμό embeddings + ευρετηρίαση διανυσμάτων)

Η επίδοσή της σε σχέση με τις κλασικές μεθόδους:
- Μπορεί να έχει παρόμοια επίδοση με το TF‑IDF σε “τυπικά” metrics, αλλά συχνά διαφέρει ως προς το τι θεωρεί σχετικό (ιδίως σε παραφράσεις/συνώνυμα)

## Πειράματα

Θα αξιολογήσουμε τις μεθόδους στο dataset **NFCorpus**, που περιέχει άρθρα PubMed και ερωτήματα σχετικά με τη διατροφή. Θα:
- Δημιουργήσουμε/φορτώσουμε το index όπως κάναμε με το TF-IDF
- Υπολογίσουμε μετρικές αξιολόγησης (NDCG@k, Recall@k)
- Συγκρίνουμε τη συμπεριφορά και τα πλεονεκτήματα/μειονεκτήματα έναντι της ανάκτησης με κλασικές μεθόδους

```
Κωνσταντίνος Καραμανής: constantine@utexas.edu
http://users.ece.utexas.edu/~cmcaram/
The University of Texas at Austin
Archimedes/Athena RC
```


### Εγκατάσταση βιβλιοθηκών

Το notebook χρησιμοποιεί ορισμένες εξειδικευμένες βιβλιοθήκες για **Information Retrieval (IR)** και επεξεργασία κειμένου. Τις πρώτες τρεις τις έχουμε δει ήδη στο προηγούμενο notebook:

- **Pyserini**: Προσφέρει εργαλεία για δημιουργία/αναζήτηση ευρετηρίων κειμένου και είναι ιδιαίτερα χρήσιμη για αναζήτηση με λέξεις‑κλειδιά, BM25 κ.λπ., που λέγεται και "sparse search".

- **BEIR** (Benchmarking IR): Benchmark για IR που παρέχει **τυποποιημένα datasets** και **μετρικές αξιολόγησης** για σύγκριση μεθόδων.

- **NLTK** (Natural Language Toolkit): Πλατφόρμα NLP για Python. Τη χρησιμοποιούμε για βασικές προεπεξεργασίες όπως tokenization και stopword removal (όπου χρειάζεται).

Θα χρειαστούμε επίσης μια βιβλιοθήκη που δεν έχουμε δει ακόμη:

- **FAISS** (Facebook AI Similarity Search): Βιβλιοθήκη για αποδοτική αναζήτηση ομοιότητας και ομαδοποίηση σε **διανυσματικές ενσωματώσεις**. Η αναζήτηση σε χώρο διανυσμάτων γίνεται μέσω της Ευκλείδειας απόστασης ή της γωνίας μεταξύ διανυσμάτων, που, όπως έχουμε συζητήσει, βασίζεται στον υπολογισμό του εσωτερικού γινομένου. Αυτή η διαδικασία έχει υλοποιηθεί αποδοτικά στην FAISS. Εδώ χρησιμοποιούμε την έκδοση CPU (`faiss-cpu`) για nearest‑neighbor αναζητήσεις σε embeddings.

Μαζί, αυτές οι βιβλιοθήκες μάς επιτρέπουν να συγκρίνουμε κλασικές προσεγγίσεις (λέξεις‑κλειδιά) με σύγχρονες **σημασιολογικές** προσεγγίσεις (embeddings + nearest neighbors).

Σημείωση: Η Pyserini/Lucene βασίζεται σε Java, οπότε σε ορισμένα περιβάλλοντα μπορεί να ζητηθεί εγκατάσταση JDK ή/και επανεκκίνηση του kernel.

Αν εμφανιστεί μήνυμα για επανεκκίνηση του runtime μετά την εγκατάσταση, πάτησε "Yes".


In [ ]:
# Colab setup for Pyserini/Lucene.
# Run this once near the top of a fresh runtime, before importing pyserini or jnius.
import os
import sys
import subprocess
import shutil
import site
from pathlib import Path

PYTHON_PACKAGES = [
    "pyserini==2.3.0",
    "faiss-cpu",
    "beir",
    "nltk",
    "sentence-transformers",
]

def run(cmd):
    print("$", " ".join(cmd))
    subprocess.check_call(cmd)

# Pyserini/Anserini uses Lucene through the JVM. Java 21 is required.
run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "openjdk-21-jdk"])

java_home = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = f"{java_home}/bin:" + os.environ["PATH"]

# Fix Colab states where Pillow is half-upgraded and PIL files are mixed.
run([sys.executable, "-m", "pip", "uninstall", "-y", "Pillow"])
for base in site.getsitepackages():
    base = Path(base)
    for pattern in ["PIL", "Pillow-*.dist-info", "pillow-*.dist-info"]:
        for path in base.glob(pattern):
            if path.exists():
                shutil.rmtree(path, ignore_errors=True)

run([
    sys.executable, "-m", "pip", "install",
    "-q", "--no-cache-dir", "--force-reinstall",
    "Pillow==11.3.0",
])

# Install dependencies before any pyserini/jnius import.
run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", *PYTHON_PACKAGES])

print("Python:", sys.version)
print("java:", shutil.which("java"))
subprocess.check_call(["java", "-version"])

# Smoke test.
import PIL
print("Pillow:", PIL.__version__, PIL.__file__)

from pyserini.search.lucene import LuceneSearcher
print("LuceneSearcher import: ok")

In [ ]:
from tqdm import tqdm
import json
import os
import math
import tempfile
import shutil

from beir import util
from beir.datasets.data_loader import GenericDataLoader
from pyserini.search.lucene import LuceneSearcher


## Φόρτωση του dataset

Φορτώνουμε πάλι το dataset **NFCorpus** από το benchmark [BEIR](https://arxiv.org/abs/2104.08663).

- Περίπου 300 ερωτήματα (queries)
- Περίπου 3600 άρθρα (documents / corpus)

### Πώς αναπαρίσταται το dataset

- `corpus`: λεξικό της μορφής `{doc_id: document}`, όπου κάθε `document` περιέχει πεδία όπως `title`/`text`.
- `queries`: λεξικό της μορφής `{query_id: query}`.
- `qrels`: τα “ground truth” ζεύγη (ερώτημα → σχετικά έγγραφα), της μορφής `{query_id: {doc_id: score, ...}}`.

Το `score` εκφράζει το επίπεδο συνάφειας (relevance). Σε αρκετά datasets είναι δυαδικό (σχετικό/μη σχετικό), αλλά μπορεί και να είναι κλιμακωτό (graded relevance).


In [ ]:
out_dir = os.path.join(os.getcwd(), "datasets")

dataset = "nfcorpus"
url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{}.zip".format(dataset)
data_path = util.download_and_unzip(url, out_dir)

corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

/content/datasets/nfcorpus.zip:   0%|          | 0.00/2.34M [00:00<?, ?iB/s]

  0%|          | 0/3633 [00:00<?, ?it/s]

In [ ]:
def calculate_metrics(qrels, results, k_values):
    """Calculate NDCG and Recall metrics for search results."""
    ndcg = {f"NDCG@{k}": 0.0 for k in k_values}
    recall = {f"Recall@{k}": 0.0 for k in k_values}

    for query_id in qrels:
        if query_id not in results:
            continue

        relevant_docs = set(qrels[query_id].keys())
        retrieved_docs = list(results[query_id].keys())

        # Calculate NDCG
        for k in k_values:
            dcg = 0
            idcg = sum((1 / math.log2(i + 2) for i in range(min(k, len(relevant_docs)))))

            for i, doc_id in enumerate(retrieved_docs[:k]):
                if doc_id in relevant_docs:
                    dcg += 1 / math.log2(i + 2)

            if idcg > 0:
                ndcg[f"NDCG@{k}"] += dcg / idcg

        # Calculate Recall
        for k in k_values:
            retrieved_relevant = len(set(retrieved_docs[:k]) & relevant_docs)
            if len(relevant_docs) > 0:
                recall[f"Recall@{k}"] += retrieved_relevant / len(relevant_docs)

    # Average the metrics
    num_queries = len(qrels)
    for k in k_values:
        ndcg[f"NDCG@{k}"] /= num_queries
        recall[f"Recall@{k}"] /= num_queries

    return ndcg, recall

In [ ]:
import re
from collections import Counter
from typing import Dict, List, Optional
import nltk
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
import math
from tqdm import tqdm

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')

### Δημιουργία inverse index

Κατασκευάζουμε ένα ανεστραμμένο ευρετήριο (inverse index) χρησιμοποιώντας τη βιβλιοθήκη Python [Pyserini](https://github.com/castorini/pyserini) (wrapper γύρω από το Lucene). Θυμίζουμε πως η Pyserini απαιτεί ένα αρχείο (συνήθως `jsonl`) όπου κάθε γραμμή αντιστοιχεί σε ένα έγγραφο και περιλαμβάνει:

- `id`: το αναγνωριστικό του εγγράφου (document ID)
- `contents`: το κείμενο του εγγράφου

Θα μετατρέψουμε το `corpus` σε αυτή τη μορφή.


In [ ]:
#making a temporary directory to store the index
tmp_dir = tempfile.mkdtemp()

# Prepare the corpus for Pyserini indexing
corpus_path = os.path.join(tmp_dir, 'corpus.jsonl')
with open(corpus_path, 'w') as f:
    for doc_id, doc in corpus.items():
        json.dump({'id': doc_id, 'contents': doc['text']}, f)
        f.write('\n')

# Index the corpus
sparse_index_dir = os.path.join(tmp_dir, 'index')

#this command indexes our documents with pyserini
os.system(f"python -m pyserini.index -collection JsonCollection -generator DefaultLuceneDocumentGenerator -threads 1 -input {tmp_dir} -index {sparse_index_dir} -storePositions -storeDocvectors -storeRaw")

0

### Ανάκτηση (Retrieval)

Τώρα θα ανακτήσουμε έγγραφα χρησιμοποιώντας το `LuceneSearcher` από την Pyserini, όπως κάναμε και στο προηγούμενο notebook. Το `LuceneSearcher`:

1. Φορτώνει το ανεστραμμένο ευρετήριο από τον δίσκο.
2. Για κάθε ερώτημα (query), “κοιτάει” τους όρους του ερωτήματος και εντοπίζει έγγραφα στα οποία εμφανίζονται αυτοί οι όροι.
3. Βαθμολογεί και ταξινομεί (ranks) τα έγγραφα με τη μέθοδο **BM25**, η οποία είναι στενά συνδεδεμένη με την ιδέα του TF‑IDF (αλλά συνήθως συμπεριφέρεται καλύτερα στην πράξη).
4. Επιστρέφει τα top‑k αποτελέσματα με φθίνουσα σειρά σκορ.


In [ ]:
sparse_searcher = LuceneSearcher(sparse_index_dir)

results = {}
for query_id, query in tqdm(queries.items(), total=len(queries), desc="Retrieving"):
    hits = sparse_searcher.search(query, k=100)

    #we will store the results in a dictionary in the form of {doc_id: score} pairs, similar to how it is stored in the ground-truth qrels object.
    results[query_id] = {hit.docid: hit.score for hit in hits}

Retrieving: 100%|██████████| 323/323 [00:03<00:00, 84.49it/s] 


## Ανάκτηση με Σημασιολογικές Ενσωματώσεις (Dense Retrieval)

**Dense/Sparse Retrieval**: Τώρα θα εξερευνήσουμε την ανάκτηση με σημασιολογικές ενσωματώσεις, που συχνά αποκαλείται **dense retrieval**. Αυτός ο όρος προέρχεται από τα διανύσματα των ενσωματώσεων, που είναι *dense*, δηλαδή οι περισσότερες συνιστώσες τους είναι μη μηδενικές. Η ανάκτηση μέσω TF-IDF/BM25 λέγεται και "sparse retrieval". Αυτός ο όρος προκύπτει από την αναπαράσταση των κειμένων, όπου κάθε κείμενο αντιστοιχεί σε ένα διάνυσμα υψηλής διάστασης (με μέγεθος όσο το λεξιλόγιο), που απλώς μετρά πόσες φορές εμφανίζεται κάθε λέξη του λεξιλογίου στο κείμενο. Ως εκ τούτου, οι περισσότερες συνιστώσες είναι 0, επειδή ένα έγγραφο περιέχει μόνο μικρό υποσύνολο του λεξιλογίου.

### Κεντρική ιδέα

- Αντί να αναζητούμε έγγραφα που μοιράζονται τις ίδιες λέξεις, αναζητούμε έγγραφα των οποίων οι ενσωματώσεις (embeddings) είναι **κοντά** στην ενσωμάτωση του ερωτήματος (query).
- Η απόσταση μετριέται με μια μετρική ομοιότητας, όπως η **cosine similarity**, που ουσιαστικά υπολογίζεται από το εσωτερικό γινόμενο.

Αυτό επιτρέπει καλύτερη συμπεριφορά σε **συνώνυμα**, **παραφράσεις** και πιο έμμεσες σημασιολογικές συσχετίσεις.


## Σημασιολογικές Ενσωματώσεις για Κείμενο

Τις ενσωματώσεις τις έχουμε συζητήσει ήδη και θα συνεχίσουμε να τις αναλύουμε σε μεγαλύτερη λεπτομέρεια στις διαλέξεις μας. Εδώ δίνουμε μια απλοποιημένη/συνοπτική ιδέα για το πώς λειτουργούν.

Οι σημασιολογικές ενσωματώσεις είναι αριθμητικές αναπαραστάσεις κειμένου που (στον βαθμό που το έχει μάθει το μοντέλο) κωδικοποιούν **σημασιολογικό νόημα**. Όπως ένα CNN μπορεί να αναπαριστά μια εικόνα με ένα feature vector (π.χ. τα 512‑διάστατα διανύσματα από το τελευταίο επίπεδο του `ResNet18`, που είδαμε σε προηγούμενα μαθήματα και notebooks), έτσι και οι ενσωματώσεις αναπαριστούν ένα κομμάτι κειμένου ως **διάνυσμα σταθερού μήκους**, όπου κείμενα με παρόμοιο νόημα τείνουν να βρίσκονται “κοντά” στον διανυσματικό χώρο — δηλαδή, να έχουν μικρή γωνία μεταξύ τους (μεγάλο εσωτερικό γινόμενο).

Στην πράξη, αυτές οι ενσωματώσεις παράγονται από μεγάλα γλωσσικά μοντέλα ή ειδικά εκπαιδευμένα μοντέλα (θα τα συζητήσουμε πιο αναλυτικά σε επόμενες ενότητες).

## Πώς λειτουργούν (διαισθητικά, χωρίς πολλές λεπτομέρειες)

1. **Διανυσματική ενσωμάτωση/αναπαράσταση**
   - Κάθε λέξη/φράση/πρόταση/έγγραφο μετατρέπεται σε ένα διάνυσμα αριθμών
   - Η διάσταση μπορεί να είναι εκατοντάδες ή χιλιάδες. Έχουμε δει ήδη παραδείγματα:  
      * 300 για το Word2Vec,
      * 512 για το ResNet18
      * Θα δούμε: 768 για το BERT
   - Κείμενα που “μοιάζουν” νοηματικά τείνουν να έχουν παρόμοιες ενσωματώσεις (θυμίζει Word2Vec, αλλά εδώ είναι συνήθως πιο ισχυρό/γενικό, διότι οι ενσωματώσεις είναι συμφραζόμενες)

2. **Σημασιολογική ομοιότητα**
   - Παρόμοια νοήματα οδηγούν σε διανύσματα με παρόμοια “κατεύθυνση”
   - Μετράμε την ομοιότητα με **cosine similarity** (γωνία μεταξύ διανυσμάτων) ή με το εσωτερικό γινόμενο
   - Π.χ. "cat" και "kitten" πρέπει να έχουν διανυσματικές ενσωματώσεις που είναι σχετικά κοντά

3. **Απόσταση στον διανυσματικό χώρο**
   - Μικρή απόσταση / μεγάλη ομοιότητα → παρόμοιο νόημα
   - Μεγάλη απόσταση / μικρή ομοιότητα → διαφορετικό νόημα


## Ενδεικτικές εφαρμογές

1. **Αναζήτηση / Ανάκτηση**
   - Βρίσκουμε έγγραφα παρόμοια με ένα query
   - Κατατάσσουμε αποτελέσματα με βάση τη σημασιολογική συνάφεια

2. **Clustering**
   - Ομαδοποίηση παρόμοιων εγγράφων
   - Ανακάλυψη θεμάτων (topics) σε συλλογές κειμένων

3. **Ταξινόμηση (classification)**
   - Πρόβλεψη κατηγορίας/ετικέτας για κείμενο
   - Π.χ. sentiment, topic, intent, κ.λπ.

## Διαφορές από παραδοσιακές μεθόδους (TF‑IDF)

1. **Σημασιολογική κατανόηση**
   - Δεν βασίζεται αποκλειστικά σε ακριβές ταίριασμα λέξεων
   - Πιάνει συνώνυμα και συναφείς έννοιες

2. **Ευελιξία**
   - Συχνά λειτουργεί καλύτερα σε παραφράσεις και διαφορετικές διατυπώσεις
   - Μπορεί να γενικεύσει σε πολλαπλές γλώσσες (ανάλογα με το embedding model)

3. **Αποδοτικότητα στους υπολογισμούς ομοιότητας**
   - Σταθερό μέγεθος διανυσμάτων, ανεξάρτητα από το μήκος του κειμένου
   - Γρήγορες πράξεις ομοιότητας (εσωτερικό γινόμενο)


In [ ]:
from beir import util
from sentence_transformers import SentenceTransformer
from pyserini.search.faiss import FaissSearcher
import faiss
import numpy as np
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### Δημιουργία ευρετηρίου (index) για σημασιολογικά διανύσματα

Όπως και στο sparse retrieval (TF-IDF), πρώτα θα δημιουργήσουμε ένα “ευρετήριο” των εγγράφων μας. Η διαφορά είναι ότι εδώ το ευρετήριο δεν είναι ανεστραμμένο ευρετήριο όρων (inverse index)· είναι ουσιαστικά μια δομή δεδομένων πάνω σε **διανύσματα**.

Θα μπορούσαμε εύκολα να υλοποιήσουμε την ανάκτηση μόνοι μας, αφού ο βασικός υπολογισμός που απαιτείται είναι το εσωτερικό γινόμενο. Αντί γι’ αυτό, επιδεικνύουμε τη χρήση της βιβλιοθήκης FAISS, η οποία έχει υλοποιήσει την ανάκτηση (και άλλες λειτουργίες) με πολύ αποδοτικό τρόπο.

Συγκεκριμένα, θα χρησιμοποιήσουμε την κλάση `IndexFlatIP` από το [FAISS](https://github.com/facebookresearch/faiss), η οποία υλοποιεί **exact nearest-neighbor search** με βάση το εσωτερικό γινόμενο.

Το `IndexFlatIP` υπολογίζει το εσωτερικό γινόμενο μεταξύ του query embedding και κάθε document embedding, και επιστρέφει τα IDs των πιο “κοντινών” εγγράφων.

### Σημείωση για cosine similarity

- Η **cosine similarity** είναι το εσωτερικό γινόμενο των κανονικοποιημένων διανυσμάτων — το έχουμε ξαναδεί στο notebook όπου συζητήσαμε το Word2Vec.
- Γι’ αυτό πολλές υλοποιήσεις κανονικοποιούν τα embeddings, ώστε το εσωτερικό γινόμενο να αντιστοιχεί απευθείας στην cosine similarity.

Σε πραγματικά μεγάλα σώματα κειμένων, το exact search μπορεί να είναι υπολογιστικά απαγορευτικό. Σε αυτή την περίπτωση χρησιμοποιούμε **approximate nearest neighbor** (ANN) ευρετήρια. Μπορείτε να διαβάσετε για τις [άλλες δομές του FAISS (π.χ. IVF, HNSW, PQ) εδώ](https://github.com/facebookresearch/faiss/wiki/Faiss-indexes).


In [ ]:
def create_faiss_index(model, corpus, index_path):
    # Encode corpus
    embeddings = []
    doc_ids = []
    for doc_id, doc in tqdm(corpus.items(), desc="Encoding corpus"):
        #return a dense embedding from a model
        embeddings.append(model.encode("passage: " + doc['text'], convert_to_tensor=True).cpu().numpy())
        doc_ids.append(doc_id)

    embeddings = np.array(embeddings).astype('float32')

    # Create FAISS index
    dimension = embeddings.shape[1]

    #creating an index object of IndexFlatIP that can store embeddings of 'dimension'.
    index = faiss.IndexFlatIP(dimension)

    #adding all of my computed embeddings into the index
    index.add(embeddings)

    #saving the index
    faiss.write_index(index, f"{index_path}/index")

    #saves the document IDs
    with open(f"{index_path}/docid", 'w') as f:
        for doc_id in doc_ids:
            f.write(f"{doc_id}\n")

    #saving the dimension
    with open(f"{index_path}/metadata.json", 'w') as f:
        json.dump({'dimension': dimension, 'type': 'flat'}, f)


### Ανάκτηση εγγράφων από το ευρετήριο FAISS

Αντίστοιχα με το TF-IDF, θα γράψουμε μια συνάρτηση `retrieve_faiss` που:

- Για κάθε query υπολογίζει τη διανυσματική ενσωμάτωση
- Κάνει αναζήτηση στο FAISS index για τους **k κοντινότερους γείτονες** (εδώ k=1000)
- Επιστρέφει/αποθηκεύει αποτελέσματα σε ένα λεξικό `results` της μορφής `{query_id: {doc_id: score, ...}}`

Το `score` εδώ είναι η τιμή ομοιότητας που χρησιμοποιεί το index (εσωτερικό γινόμενο ή cosine similarity, ανάλογα με το πώς έχουμε ρυθμίσει/κανονικοποιήσει τις ενσωματώσεις).

Στη συνέχεια θα χρησιμοποιήσουμε το `results` για να υπολογίσουμε μετρικές αξιολόγησης (NDCG@k, Recall@k) με βάση τα `qrels`.


In [ ]:
def retrieve_faiss(model, index_path, queries):

  #load the index which contains document embeddings
  index = faiss.read_index(f"{index_path}/index")
  doc_ids = []

  #load document IDs
  with open(f"{index_path}/docid", 'r') as f:
    for line in f:
      doc_ids.append(line.strip())

  results = {}

  for query_id, query in tqdm(queries.items(), desc="Retrieving"):
    query_embedding = model.encode("query: " + query, convert_to_tensor=True).cpu().numpy()

    #find 1000 closest documents based on inner product similarity
    distances, indices = index.search(np.array([query_embedding]), 1000)

    #store the doc_id of each retrieved document along with its score i.e. the value of the inner product.
    results[query_id] = {doc_ids[i]: distances[0][j] for j, i in enumerate(indices[0])}

  return results


### Σύγκριση μοντέλων embeddings

Θα συγκρίνουμε **τρία έτοιμα (off‑the‑shelf) μοντέλα** για σημασιολογική ανάκτηση.

- [bert-base-uncased](https://huggingface.co/google-bert/bert-base-uncased)
  - Είναι ένα [masked language model](https://huggingface.co/docs/transformers/en/tasks/masked_language_modeling) γενικού σκοπού.
  - **Δεν** έχει εκπαιδευτεί ρητά για ανάκτηση. Αν το χρησιμοποιήσουμε “ως έχει” για embeddings, συχνά θα δούμε χειρότερη κατάταξη σε σχέση με retrieval‑trained μοντέλα.

- [all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)
  - Είναι μοντέλο Sentence‑Transformers, εκπαιδευμένο ώστε τα embeddings να είναι χρήσιμα για σημασιολογική ανάκτηση.
  - Χρησιμοποιεί στόχο εκπαίδευσης που μεγιστοποιεί την ομοιότητα (query, relevant doc) και ελαχιστοποιεί την ομοιότητα (query, irrelevant doc).
  - Αυτή η οικογένεια στόχων λέγεται **contrastive loss**, την οποία θα συζητήσουμε εκτενώς στις επόμενες διαλέξεις.
  - Το **InfoNCE** είναι από τις πιο συχνές επιλογές loss σε retrieval/representation learning.

- [e5-small-v2](https://huggingface.co/intfloat/e5-small-v2)
  - Επίσης εκπαιδεύεται με contrastive learning, αλλά πάνω σε **πολύ μεγάλης κλίμακας** δεδομένα (training examples mined από το διαδίκτυο).
  - Το paper τους περιγράφει λεπτομερώς τη διαδικασία εκπαίδευσης.

Στόχος μας είναι να δούμε ότι η απόδοση της σημασιολογικής ανάκτησης εξαρτάται καθοριστικά από το μοντέλο που χρησιμοποιούμε και από το αν είναι κατάλληλα εκπαιδευμένο για ανάκτηση (και όχι απλώς για language modeling). Σημειώνουμε επίσης πως δεν υπάρχει ένα μοντέλο σημασιολογικής ενσωμάτωσης ή ανάκτησης που υπερισχύει όλων των άλλων. Μπορεί ένα μοντέλο να έχει καλύτερη επίδοση σε ένα σώμα κειμένων και χειρότερη σε κάποιο άλλο. [Μπορείτε να δείτε εδώ](https://huggingface.co/spaces/mteb/leaderboard) σύγκριση επιδόσεων διαφόρων μοντέλων.


## Hugging Face

Θα κατεβάσουμε και τα τρία μοντέλα από το [Hugging Face](https://huggingface.co). Το Hugging Face είναι μια πλατφόρμα και συγχρόνως μια κοινότητα για τεχνητή νοημοσύνη και μηχανική μάθηση. Περιλαμβάνει πολλά μοντέλα, σύνολα δεδομένων και βιβλιοθήκες όπως οι Transformers, τις οποίες θα χρησιμοποιήσουμε εκτενώς, και επιτρέπει εύκολη ενσωμάτωση στην Python.


In [ ]:
!pip install -q huggingface_hub[hf_xet]

In [ ]:
# Clear GPU cache if CUDA is available
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"CUDA available. GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory before loading: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

    # If GPU memory is low, force CPU usage
    if torch.cuda.memory_allocated(0) / 1024**3 > 0.5:  # More than 0.5GB already used
        print("Warning: GPU memory appears occupied. Consider restarting kernel or using CPU.")
        device = 'cpu'
    else:
        # Use 'cuda' which defaults to 'cuda:0'
        device = 'cuda'
else:
    device = 'cpu'
    print("CUDA not available, using CPU")

print(f"Using device: {device}")
# Load the model
model = SentenceTransformer('bert-base-uncased', device=device)  # Load a pretrained embedding model
# model = SentenceTransformer('all-MiniLM-L6-v2', device=device)  # You can swap for other models
# model = SentenceTransformer('intfloat/e5-small-v2', device=device)  # <-- THE ERROR: colons are invalid here! Just use (model_name, device=device)


CUDA available. GPU: Tesla T4
GPU memory before loading: 0.00 GB
Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Ένα Μικρό Πείραμα

Έχουμε φορτώσει το μοντέλο μας. Τώρα θα δούμε ένα απλό πείραμα που αναφέραμε στο τέλος του [Word2Vec notebook](https://colab.research.google.com/drive/1dfmCgiNyNoxnam-JxfoPrQWhYWfSvDox?usp=drive_link)

```python
toy_corpus = [
    "The Parthenon sits atop the Acropolis as a symbol of ancient Greek and Athenian civilization.",
    "Greek olive oil is considered among the finest in the world.",
    "Mount Olympus was believed to be the home of the Greek deities.",
    "The island of Santorini is known for its white buildings and caldera views.",
    "Greece has a long tradition of philosophy, with thinkers like Socrates and Plato.",
    "The euro is the official currency used throughout Greece.",
    "Greek cuisine features dishes like moussaka, souvlaki, and spanakopita.",
    "The Battle of Marathon marked a key victory for the Greeks against the Persians.",
    "Thessaloniki is a vibrant city in northern Greece known for its history and nightlife.",
    "The Greek alphabet has been in use for nearly three millennia.",
    "Many tourists visit Greece each year for its beaches, islands, and ancient ruins.",
    "Democracy originated in classical Athens around the 5th century BCE.",
    "Greek Orthodox Christianity is the predominant religion in Greece.",
    "Crete is the largest Greek island and was once home to the Minoan civilization.",
    "Athens hosted the first modern Olympic Games in 1896.",
    "Greek shipping companies are influential in the global maritime industry.",
    "The Delphi oracle was believed to deliver messages from the god Apollo.",
    "Greek mythology has had a lasting impact on Western literature and art.",
    "Rhodes is an island famous for its medieval Old Town and ancient ruins.",
    "Greece’s national day is celebrated on March 25th, commemorating independence from Ottoman rule."
]

query = "Where did the ancient gods of Greek legends supposedly reside?"
```


In [ ]:
# Define query and toy_corpus
toy_corpus = [
    "The Parthenon sits atop the Acropolis as a symbol of ancient Greek and Athenian civilization.",
    "Greek olive oil is considered among the finest in the world.",
    "Mount Olympus was believed to be the home of the Greek deities.",
    "The island of Santorini is known for its white buildings and caldera views.",
    "Greece has a long tradition of philosophy, with thinkers like Socrates and Plato.",
    "The euro is the official currency used throughout Greece.",
    "Greek cuisine features dishes like moussaka, souvlaki, and spanakopita.",
    "The Battle of Marathon marked a key victory for the Greeks against the Persians.",
    "Thessaloniki is a vibrant city in northern Greece known for its history and nightlife.",
    "The Greek alphabet has been in use for nearly three millennia.",
    "Many tourists visit Greece each year for its beaches, islands, and ancient ruins.",
    "Democracy originated in classical Athens around the 5th century BCE.",
    "Greek Orthodox Christianity is the predominant religion in Greece.",
    "Crete is the largest Greek island and was once home to the Minoan civilization.",
    "Athens hosted the first modern Olympic Games in 1896.",
    "Greek shipping companies are influential in the global maritime industry.",
    "The Delphi oracle was believed to deliver messages from the god Apollo.",
    "Greek mythology has had a lasting impact on Western literature and art.",
    "Rhodes is an island famous for its medieval Old Town and ancient ruins.",
    "Greece’s national day is celebrated on March 25th, commemorating independence from Ottoman rule."
]

query = "Where did the ancient gods of Greek legends supposedly reside?"

# Encode query and candidates
query_embedding = model.encode(query, normalize_embeddings=True)  # Normalize for cosine similarity
corpus_embeddings = [model.encode(c, normalize_embeddings=True) for c in toy_corpus]

### Πώς μοιάζουν οι ενσωματώσεις;

1. Είναι διανύσματα σε 768 διαστάσεις: μία πρόταση --> ένα διάνυσμα
2. Οι ενσωματώσεις (τα διανύσματα), όπως και στο Word2Vec, έχουν μη μηδενικές συνιστώσες, και έτσι λέγονται και "dense embeddings".


In [ ]:
query_embedding.shape

(768,)

In [ ]:
corpus_embeddings[0].shape

(768,)

In [ ]:
print(query_embedding.shape)
print(corpus_embeddings[0].shape)
query_embedding

(768,)
(768,)


array([ 5.79694100e-02,  5.12123369e-02, -5.63923605e-02, -4.42793267e-03,
        1.90029163e-02, -2.78035849e-02,  5.73729500e-02,  8.76795426e-02,
       -4.39798599e-03,  1.27397641e-03,  2.42921486e-02, -3.84102538e-02,
       -4.14184146e-02,  6.82746097e-02, -3.95451635e-02,  2.10565645e-02,
       -9.41088609e-03,  5.13839498e-02, -3.98181304e-02,  6.18512705e-02,
        3.67451226e-04,  1.91909820e-02, -1.58653725e-02,  3.97883058e-02,
        3.61036770e-02,  2.34835465e-02, -1.39712309e-02,  8.49253405e-03,
       -2.08188370e-02,  3.28630544e-02,  1.53512340e-02, -5.37787704e-03,
       -6.47711977e-02, -2.81135552e-02, -1.02914143e-02, -1.20246969e-02,
       -1.51253222e-02,  8.14211275e-03, -1.48303444e-02,  5.10580577e-02,
       -4.68632653e-02, -6.77676350e-02,  4.34411056e-02,  4.74177487e-02,
       -2.25749463e-02, -2.64763683e-02,  2.06090771e-02,  2.63822656e-02,
        7.56894611e-03, -9.32063069e-03, -3.53698363e-03,  2.90163811e-02,
       -3.65957767e-02,  

### Σημασιολογική Ομοιότητα

Εφόσον είναι κανονικοποιημένες ώστε να έχουν μήκος 1 (normalized), η ομοιότητα αντιστοιχεί στο εσωτερικό γινόμενο, που στην Python υπολογίζεται με την εντολή `np.dot(x, y)`.

Υπολογίζουμε την ομοιότητα της ερώτησης (query) με κάθε πρόταση. Παρατηρούμε πως η πρόταση που κρίνει το μοντέλο πως έχει την μέγιστη ομοιότητα δεν είναι αυτή που έχει την μεγαλύτερη επικάλυψη.


In [ ]:
# Compute cosine similarities
similarities = [np.dot(query_embedding, cand_emb) for cand_emb in corpus_embeddings]

# Combine corpus and similarities, then sort in decreasing order of similarity
sorted_results = sorted(zip(toy_corpus, similarities), key=lambda x: x[1], reverse=True)

# Display results
print('Query:', query)
print("\nDocuments sorted by decreasing similarity:")
for cand, sim in sorted_results:
    print(f"Cosine similarity between Query and '{cand}': {sim:.4f}")


Query: Where did the ancient gods of Greek legends supposedly reside?

Documents sorted by decreasing similarity:
Cosine similarity between Query and 'Mount Olympus was believed to be the home of the Greek deities.': 0.8096
Cosine similarity between Query and 'Crete is the largest Greek island and was once home to the Minoan civilization.': 0.7426
Cosine similarity between Query and 'The Delphi oracle was believed to deliver messages from the god Apollo.': 0.7380
Cosine similarity between Query and 'The Parthenon sits atop the Acropolis as a symbol of ancient Greek and Athenian civilization.': 0.7260
Cosine similarity between Query and 'Greek mythology has had a lasting impact on Western literature and art.': 0.7067
Cosine similarity between Query and 'Many tourists visit Greece each year for its beaches, islands, and ancient ruins.': 0.7036
Cosine similarity between Query and 'Greece has a long tradition of philosophy, with thinkers like Socrates and Plato.': 0.6753
Cosine similarity 

## Ας ξαναδούμε το απλό μας πρόβλημα

Ας θυμηθούμε το διδακτικό παράδειγμα από το προηγούμενο notebook: ένα query μπορεί να περιέχει κάποιες “θορυβώδεις” λέξεις που εμφανίζονται παντού (π.χ. *treatment*, *interaction*, *metformin*) και να δυσκολεύει μια καθαρά keyword‑based μέθοδο να ξεχωρίσει το πραγματικά σχετικό έγγραφο.

Εδώ θα ξανατρέξουμε το ίδιο σενάριο με σημασιολογική αναζήτηση:

- Θα υπολογίσουμε embeddings για το query και για όλα τα έγγραφα
- Θα μετρήσουμε ομοιότητα (cosine similarity)
- Θα δούμε σε ποια θέση (rank) εμφανίζεται το σωστό έγγραφο που μιλάει συγκεκριμένα για την αλληλεπίδραση **cimetidine–metformin**

Η προσδοκία είναι ότι το dense retrieval θα “καταλάβει” τη σχέση των ονομάτων φαρμάκων και της έννοιας *interaction*, ακόμη κι αν η διατύπωση στα έγγραφα δεν ταιριάζει λέξη προς λέξη με το query.


In [ ]:
toy_corpus2 = [
"Clinic newsletter: patients with type 2 diabetes were reminded that medication, diet, and exercise work together; the article mentions metformin as a common start and notes that any new drug should prompt an interaction check during treatment review.",
"Follow-up note: the nurse reviewed side effects and asked about supplements, alcohol, and prescriptions, explaining that interaction screening matters even for familiar medicines like metformin when treatment plans change.",
"Educational summary: metformin is often first-line; the piece repeats that clinicians monitor kidney function and ask about interaction risks, because dehydration or other drugs can alter metformin safety in routine treatment.",
"Research digest: a small study of patients on metformin discussed adherence and lifestyle; it briefly mentions interaction concerns but focuses on diet, sleep, and long-term treatment goals.",
"Discharge instructions: continue metformin with meals, record glucose, and bring a medication list; the sheet says ‘tell us about any interaction with new pills’ as part of safe treatment planning.",
"Review paragraph: metformin lowers hepatic glucose output; the author adds that clinicians consider renal function and interaction flags, especially when treatment includes multiple agents.",
"Pharmacy handout: before refills, ask about interaction questions; it lists common warnings (alcohol, dehydration, contrast dye) and says metformin is usually well tolerated when treatment is stable.",
"Patient story: after starting metformin, the patient improved A1C; the narrative mentions an interaction screening at each visit but is mostly about motivation, meal planning, and staying on treatment.",
"Guideline excerpt: reassess treatment every few months; check labs, reinforce activity, and review interaction lists; metformin remains a baseline drug for many patients.",
"Clinician memo: lactic acidosis is rare; avoid metformin during acute illness; the memo repeats that interaction review is routine but does not describe any mechanism.",
"FAQ page: how metformin works, how to take metformin, and why metformin continues; it includes a generic line about interaction checks during treatment but no examples.",
"Community brochure: programs support patients with reminders, coaching, and medication literacy; it encourages asking pharmacists about interaction risks when taking metformin and other drugs.",
"Case narrative: missed appointments complicated treatment; the clinician emphasized consistent metformin dosing and repeated interaction checks as new prescriptions were added over time.",
"Imaging note: pause metformin around iodinated contrast; the note frames this as an interaction-adjacent precaution tied to kidney stress and follow-up testing after treatment.",
"Lecture transcript: ‘interaction’ is a broad pharmacology term; the speaker mentions metformin in passing and stresses that interaction questions should be documented for patient safety.",
"Pharmacokinetic note (the one we want): cimetidine reduces renal tubular secretion, increasing plasma levels; monitoring recommended when co-administered.",
"Abstract: the study compared two treatment groups; it uses the word interaction once and metformin once, but otherwise stays generic about outcomes in patients.",
"Clinic checklist: confirm metformin, confirm allergies, and ask about interaction with any new medication; the checklist is repetitive but not specific about which drug interacts.",
"Call center script: if patients report a new prescription, staff log a possible interaction and route the metformin question to pharmacy, but the script does not mention cimetidine or any mechanism.",
"Medication safety guide: always check for metformin interaction before prescribing new drugs; metformin interaction risks must be assessed; patients on metformin need interaction screening; metformin interaction protocols should be followed; never overlook potential metformin interaction.",
"Treatment protocol document: standard procedure involves checking metformin interaction with all concurrent medications; metformin interaction databases should be consulted; any metformin interaction concerns require immediate review; metformin interaction warnings are critical for patient safety.",
"Clinical reminder bulletin: metformin interaction metformin interaction metformin interaction—this bulletin emphasizes metformin interaction monitoring; staff must document every metformin interaction check; patient records must show metformin interaction assessment; all metformin interaction reports require follow-up.",
"Hospital policy manual: section on metformin interaction management; metformin interaction screening is mandatory; metformin interaction documentation requirements; metformin interaction reporting procedures; metformin interaction training modules; comprehensive metformin interaction guidelines.",
"Pharmacy training manual: metformin interaction protocols; metformin interaction safety checks; metformin interaction review process; metformin interaction documentation standards; metformin interaction assessment tools; metformin interaction guidelines for pharmacists.",
"Quality assurance checklist: verify metformin interaction screening completed; confirm metformin interaction documentation present; ensure metformin interaction alerts reviewed; check metformin interaction flags addressed; validate metformin interaction protocols followed.",
"Standard operating procedure: step one check metformin interaction; step two document metformin interaction; step three review metformin interaction; step four report metformin interaction; all metformin interaction steps mandatory."
]
toy_query2 = "I want to learn about cimetidine metformin interaction"


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Encode the query
print("=" * 80)
print("Semantic (Dense) Search Results")
print("=" * 80)
print(f"\nQuery: '{toy_query2}'")
print(f"Encoding query and {len(toy_corpus2)} documents...")

query_embedding = model.encode(toy_query2, normalize_embeddings=True, show_progress_bar=False)

# Encode all documents in the corpus
doc_embeddings = model.encode(toy_corpus2, normalize_embeddings=True, show_progress_bar=True)

# Calculate cosine similarity between query and all documents
similarities = cosine_similarity([query_embedding], doc_embeddings)[0]

# Get top 10 results
top_k = 10
top_indices = np.argsort(similarities)[::-1][:top_k]

print(f"\nTop {top_k} Semantic Search Results (by cosine similarity):")
print("-" * 80)

for rank, doc_idx in enumerate(top_indices, 1):
    similarity_score = similarities[doc_idx]
    doc_preview = toy_corpus2[doc_idx][:80] + "..." if len(toy_corpus2[doc_idx]) > 80 else toy_corpus[doc_idx]
    print(f"{rank}. [Score: {similarity_score:.4f}] Doc {doc_idx}: {doc_preview}")

# Find and highlight the relevant document (contains cimetidine)
cimetidine_doc_idx = None
for i, doc in enumerate(toy_corpus2):
    if "cimetidine" in doc.lower():
        cimetidine_doc_idx = i
        break

print(f"\n✓ The actual relevant document about cimetidine-metformin interaction")
if cimetidine_doc_idx is not None:
    # Calculate rank from all similarities
    all_ranks = np.argsort(similarities)[::-1]
    doc_rank = np.where(all_ranks == cimetidine_doc_idx)[0][0] + 1
    print(f"  Document (Doc {cimetidine_doc_idx}) appears at rank {doc_rank}")
    print(f"  Similarity score: {similarities[cimetidine_doc_idx]:.4f}")
    if doc_rank <= top_k:
        print(f"  ✓ It's in the top {top_k}! (Semantic search successfully found the relevant document)")
    else:
        print(f"  ⚠️  It's not in the top {top_k}, but still better ranked than many non-relevant docs")
else:
    print(f"  ⚠️  Could not find document containing 'cimetidine'")



Semantic (Dense) Search Results

Query: 'I want to learn about cimetidine metformin interaction'
Encoding query and 26 documents...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Top 10 Semantic Search Results (by cosine similarity):
--------------------------------------------------------------------------------
1. [Score: 0.7768] Doc 19: Medication safety guide: always check for metformin interaction before prescribi...
2. [Score: 0.7749] Doc 15: Pharmacokinetic note (the one we want): cimetidine reduces renal tubular secreti...
3. [Score: 0.7656] Doc 23: Pharmacy training manual: metformin interaction protocols; metformin interaction...
4. [Score: 0.7638] Doc 10: FAQ page: how metformin works, how to take metformin, and why metformin continue...
5. [Score: 0.7469] Doc 20: Treatment protocol document: standard procedure involves checking metformin inte...
6. [Score: 0.7417] Doc 22: Hospital policy manual: section on metformin interaction management; metformin i...
7. [Score: 0.7409] Doc 25: Standard operating procedure: step one check metformin interaction; step two doc...
8. [Score: 0.7393] Doc 14: Lecture transcript: ‘interaction’ is a broad pharmacology t

### Συμπέρασμα

Χαρακτηριστικά της Σημασιολογικής (Semantic) Αναζήτησης:
1. Χρησιμοποιεί συμπαγείς διανυσματικές αναπαραστάσεις που έχουν μάθει νευρωνικά δίκτυα
2. Μπορεί να ταιριάξει έγγραφα με βάση το νόημα, όχι μόνο με ακριβείς αντιστοιχίσεις λέξεων
3. Αντιμετωπίζει φυσικά τα συνώνυμα και τις σχετικές έννοιες
4. Μπορεί να κάνει διαφορετική κατάταξη από τη μέθοδο TF-IDF ή το BoW, λόγω της κατανόησης του νοήματος



## NFCorpus

Επιστρέφουμε τώρα στο NFCorpus και συγκρίνουμε τα 3 μοντέλα που κατεβάσαμε από το Hugging Face, με τις μετρικές αξιολόγησης NDCG και Recall.

Παρατηρούμε πως το 2ο και το 3ο μοντέλο είναι πολύ καλύτερα. Σε αντίθεση με το `bert-base-uncased`, αυτά τα μοντέλα έχουν εκπαιδευτεί/προσαρμοστεί (fine-tuned) για καλύτερη επίδοση σε προβλήματα ανάκτησης. Θα δούμε στις επόμενες διαλέξεις πώς γίνεται αυτή η εκπαίδευση.


In [ ]:
import os
import tempfile
import shutil
from sentence_transformers import SentenceTransformer

# Models to evaluate
models = {
    'bert-base-uncased': 'bert',
    'all-MiniLM-L6-v2': 'minilm',
    'intfloat/e5-small-v2': 'e5'
}

all_results = {}
index_dirs = {}  # Store index paths for later use

for model_name, short_name in models.items():
    print(f"\nEvaluating model: {model_name}")

    # Load the model
    model = SentenceTransformer(model_name, device=device)

    # Create a unique FAISS index directory for this model
    dense_index_dir = f"dense_index_dir_{short_name}"
    os.makedirs(dense_index_dir, exist_ok=True)  # Ensure directory exists

    # Store index directory for reference
    index_dirs[model_name] = dense_index_dir

    # Create FAISS index
    create_faiss_index(model, corpus, dense_index_dir)

    # Retrieve and Evaluate results
    results = retrieve_faiss(model, dense_index_dir, queries)
    k_values = [10, 100]
    ndcg, recall = calculate_metrics(qrels, results, k_values)

    all_results[model_name] = {
        "NDCG@10": ndcg["NDCG@10"],
        "Recall@100": recall["Recall@100"]
    }

# Print stored index directories
print("\nCreated FAISS indices:")
for model_name, index_dir in index_dirs.items():
    print(f"{model_name}: {index_dir}")



Evaluating model: bert-base-uncased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Retrieving: 100%|██████████| 323/323 [00:03<00:00, 82.65it/s]



Evaluating model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Retrieving: 100%|██████████| 323/323 [00:03<00:00, 90.37it/s]


Created FAISS indices:
bert-base-uncased: dense_index_dir_bert
all-MiniLM-L6-v2: dense_index_dir_minilm
intfloat/e5-small-v2: dense_index_dir_e5


In [ ]:
for model_name, metrics in all_results.items():
    print(f"\n<=Model: {model_name}")
    print(f"NDCG@10: {metrics['NDCG@10']}")
    print(f"Recall@100: {metrics['Recall@100']}")
#


<=Model: bert-base-uncased
NDCG@10: 0.025247324098459326
Recall@100: 0.07644786176492853

<=Model: all-MiniLM-L6-v2
NDCG@10: 0.3084619012045983
Recall@100: 0.3000657848155424

<=Model: intfloat/e5-small-v2
NDCG@10: 0.314266857840424
Recall@100: 0.27896551207740794


In [ ]:
sparse_searcher = LuceneSearcher(sparse_index_dir)

results = {}
for query_id, query in tqdm(queries.items(), total=len(queries), desc="Retrieving"):
    hits = sparse_searcher.search(query, k=100)

    #we will store the results in a dictionary in the form of {doc_id: score} pairs, similar to how it is stored in the ground-truth qrels object.
    results[query_id] = {hit.docid: hit.score for hit in hits}

Retrieving: 100%|██████████| 323/323 [00:01<00:00, 187.48it/s]


In [ ]:
# Configuration
k_values = [5, 10, 20, 100]

# Calculate metrics for TF-IDF search
tfidf_ndcg, tfidf_recall = calculate_metrics(qrels, results, k_values)

# Display results
print("\nBM25 Search Metrics:")
for k in k_values:
    print(f"NDCG@{k}: {tfidf_ndcg[f'NDCG@{k}']:.4f}")
    print(f"Recall@{k}: {tfidf_recall[f'Recall@{k}']:.4f}")


BM25 Search Metrics:
NDCG@5: 0.3505
Recall@5: 0.1198
NDCG@10: 0.3131
Recall@10: 0.1504
NDCG@20: 0.2805
Recall@20: 0.1756
NDCG@100: 0.2578
Recall@100: 0.2415
